# Length-Aware Category Batching — Proxy Benchmark & Data-Driven Bands
Clean pipeline notebook. Two data paths, clearly separated:
- **Path A — REAL datasets** (HuggingFace, via `datasets_real.json`)
- **Path B — SYNTHETIC datasets** (generated locally, no network)

Run **Section 0-1 once**, then pick **Path A or Path B** for data, then run the
shared sections (bands -> benchmark). Thanks to load-time bands, changing cutoffs
no longer requires rebuilding shards.

## 0 - Clone & locate the project

In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [3]:
import os, sys, glob, shutil, subprocess

REPO_URL = "https://github.com/SniAssia/sft.git"
shutil.rmtree("/content/sft", ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/sft"], check=True)

def find_code_dir():
    for root, dirs, files in os.walk("/content"):
        dirs[:] = [d for d in dirs if d != ".git"]
        if "bindings.cpp" in files:
            return root
    return None

CODE = find_code_dir()
assert CODE, "bindings.cpp not found"
sys.path.insert(0, CODE)
print("code dir:", CODE)

code dir: /content/sft/sft_final


## 1 - Build the C++ extension (once)\nRebuild only when C++ changes. Cutoff changes do NOT need this anymore.

In [4]:
b = subprocess.run([sys.executable, "setup.py", "build_ext", "--inplace"],
                   cwd=CODE, capture_output=True, text=True)
print(b.stdout[-800:])
if b.returncode != 0:
    print("STDERR:\n", b.stderr[-2500:]); raise RuntimeError("build failed")
import torch
import uds_loader
print("uds_loader OK:", [x for x in dir(uds_loader) if not x.startswith("_")])

64-cpython-312/content/sft/sft_final/bindings.o -O3 -std=c++17 -fvisibility=hidden -DTORCH_API_INCLUDE_EXTENSION_H -DTORCH_EXTENSION_NAME=uds_loader
creating build/lib.linux-x86_64-cpython-312
x86_64-linux-gnu-g++ -fno-strict-overflow -Wsign-compare -DNDEBUG -g -O2 -Wall -g -fstack-protector-strong -Wformat -Werror=format-security -g -fwrapv -O2 -shared -Wl,-O1 -Wl,-Bsymbolic-functions -Wl,-Bsymbolic-functions -g -fwrapv -O2 build/temp.linux-x86_64-cpython-312/content/sft/sft_final/bindings.o -L/usr/local/lib/python3.12/dist-packages/torch/lib -L/usr/lib/x86_64-linux-gnu -lc10 -ltorch -ltorch_cpu -ltorch_python -o build/lib.linux-x86_64-cpython-312/uds_loader.cpython-312-x86_64-linux-gnu.so -lpthread
copying build/lib.linux-x86_64-cpython-312/uds_loader.cpython-312-x86_64-linux-gnu.so -> 

uds_loader OK: ['CollatedPool', 'DataPipeline', 'PipelineConfig']


---
# PATH A - REAL datasets
Uses `datasets_real.json` (HF sources). Needs an HF token. Skip to Path B if HF fails.

In [5]:
from huggingface_hub import login
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None
if HF_TOKEN:
    login(token=HF_TOKEN); print("HF authenticated")
else:
    print("no HF_TOKEN — set it in Colab Secrets, or use Path B")

HF authenticated


In [6]:
# restore/inspect the REAL HF dataset config (edit as needed)
import json, os
real_cfg = [
  {"name":"alpaca","path":"tatsu-lab/alpaca","source":"hf","format":"alpaca","split":"train","limit":None},
  {"name":"ultrachat","path":"HuggingFaceH4/ultrachat_200k","source":"hf","format":"chat","split":"train_sft","limit":None},
  {"name":"openorca","path":"Open-Orca/OpenOrca","source":"hf","format":"pair","split":"train",
   "streaming":True,"limit":200000,"prompt_field":"question","response_field":"response"},
]
json.dump(real_cfg, open(os.path.join(CODE,"datasets_real.json"),"w"), indent=2)
print(json.dumps(real_cfg, indent=2))

[
  {
    "name": "alpaca",
    "path": "tatsu-lab/alpaca",
    "source": "hf",
    "format": "alpaca",
    "split": "train",
    "limit": null
  },
  {
    "name": "ultrachat",
    "path": "HuggingFaceH4/ultrachat_200k",
    "source": "hf",
    "format": "chat",
    "split": "train_sft",
    "limit": null
  },
  {
    "name": "openorca",
    "path": "Open-Orca/OpenOrca",
    "source": "hf",
    "format": "pair",
    "split": "train",
    "streaming": true,
    "limit": 200000,
    "prompt_field": "question",
    "response_field": "response"
  }
]


In [7]:
import os
p = os.path.join(CODE, "prepare_shards.py")
lines = open(p).read().splitlines()
seen, out = set(), []
for ln in lines:
    key = None
    if 'add_argument("--band-short-max"' in ln:  key = "short"
    if 'add_argument("--band-medium-max"' in ln: key = "medium"
    if key and key in seen:
        continue                      # drop the duplicate
    if key: seen.add(key)
    out.append(ln)
open(p, "w").write("\n".join(out) + "\n")

# verify: each should appear exactly once
import subprocess
print(subprocess.run(["grep","-n","band-short-max\\|band-medium-max", p],
                     capture_output=True, text=True).stdout)

632:    ap.add_argument("--band-short-max", type=int, default=512)
633:    ap.add_argument("--band-medium-max", type=int, default=1536)



In [8]:
# Build shards from REAL datasets (token passed into the SUBPROCESS env)
import json, os, subprocess
MAXLEN = 1024
SH = os.path.join(CODE, "_shards")
env = dict(os.environ)
if HF_TOKEN:
    env["HF_TOKEN"] = HF_TOKEN; env["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
env["HF_HUB_DISABLE_XET"] = "1"
r = subprocess.run([sys.executable, "prepare_shards.py",
    "--config", "datasets_real.json", "--out", SH,
    "--tokenizer", "inceptionai/jais-family-590m",
    "--max-seq-length", str(MAXLEN), "--shard-size", "1024", "--workers", "4"],
    cwd=CODE, capture_output=True, text=True, env=env)
print(r.stdout[-2000:]); print("STDERR tail:\n", r.stderr[-1500:])
if r.returncode != 0: raise RuntimeError("shard prep failed")
meta = json.load(open(os.path.join(SH, "meta.json")))
print("\nmeta:", {k: meta[k] for k in ["vocab_size","pad_id","max_seq_length","num_samples","band_counts","source_stats"]})

al/_shards/shard_00426.bin  (1024 samples)
[shard] wrote /content/sft/sft_final/_shards/shard_00427.bin  (1024 samples)
[shard] wrote /content/sft/sft_final/_shards/shard_00428.bin  (1024 samples)
[shard] wrote /content/sft/sft_final/_shards/shard_00429.bin  (1024 samples)
[source:openorca] read=200000 kept=200000
[shard] wrote /content/sft/sft_final/_shards/shard_00430.bin  (1024 samples)
[shard] wrote /content/sft/sft_final/_shards/shard_00431.bin  (1024 samples)
[shard] wrote /content/sft/sft_final/_shards/shard_00432.bin  (1024 samples)
[shard] wrote /content/sft/sft_final/_shards/shard_00433.bin  (1024 samples)
[shard] wrote /content/sft/sft_final/_shards/shard_00434.bin  (1024 samples)
[shard] wrote /content/sft/sft_final/_shards/shard_00435.bin  (1024 samples)
[shard] wrote /content/sft/sft_final/_shards/shard_00436.bin  (1024 samples)
[shard] wrote /content/sft/sft_final/_shards/shard_00437.bin  (1024 samples)
[shard] wrote /content/sft/sft_final/_shards/shard_00438.bin  (1024 

---
# PATH B - SYNTHETIC datasets (no network)
Generates 3 local JSONL datasets (60k/20k/200k) with controlled length spread.

In [ ]:
# generate synthetic datasets locally
import subprocess, sys, os, glob
r = subprocess.run([sys.executable, "gen_datasets.py"], cwd=CODE, capture_output=True, text=True)
print("STDOUT:", r.stdout); print("return:", r.returncode)
print("files:", glob.glob(os.path.join(CODE, "ds_*.jsonl")))

In [ ]:
# point config at the synthetic files
import json, os, glob
found = {os.path.basename(p): p for p in glob.glob(os.path.join(CODE, "**", "ds_*.jsonl"), recursive=True)}
cfg = [
  {"name":"small","path":found["ds_small_60k.jsonl"],"source":"jsonl","format":"alpaca","limit":None},
  {"name":"tiny", "path":found["ds_tiny_20k.jsonl"], "source":"jsonl","format":"alpaca","limit":None},
  {"name":"big",  "path":found["ds_big_200k.jsonl"], "source":"jsonl","format":"alpaca","limit":None},
]
json.dump(cfg, open(os.path.join(CODE, "datasets_real.json"), "w"), indent=2)
print("all exist:", all(os.path.exists(d["path"]) for d in cfg))

In [ ]:
# Build shards from SYNTHETIC datasets (no token needed)
import json, os, subprocess
MAXLEN = 1024
SH = os.path.join(CODE, "_shards")
r = subprocess.run([sys.executable, "prepare_shards.py",
    "--config", "datasets_real.json", "--out", SH,
    "--tokenizer", "inceptionai/jais-family-590m",
    "--max-seq-length", str(MAXLEN), "--shard-size", "1024", "--workers", "4"],
    cwd=CODE, capture_output=True, text=True)
print(r.stdout[-2000:])
if r.returncode != 0:
    print("STDERR:\n", r.stderr[-3000:]); raise RuntimeError("shard prep failed")
meta = json.load(open(os.path.join(SH, "meta.json")))
print("\nmeta:", {k: meta[k] for k in ["vocab_size","pad_id","max_seq_length","num_samples","band_counts","source_stats"]})

---
# SHARED - Data-driven bands & benchmark
Run after EITHER Path A or Path B has built shards into `_shards/`.

## 2 - Verify the length histogram is present

In [9]:
import json, os
meta = json.load(open(os.path.join(SH, "meta.json")))
print("histogram present:", "length_histogram" in meta,
      "| distinct lengths:", len(meta.get("length_histogram", {})))
assert "length_histogram" in meta, "rebuild shards with the histogram change first"

histogram present: True | distinct lengths: 1010


## 3 - Derive optimal band cutoffs from the corpus

In [10]:
import subprocess, sys, json, os
WINDOW, B = 4, 32

r = subprocess.run([sys.executable, "derive_bands.py",
    "--meta", os.path.join(SH, "meta.json"),
    "--resident-window", str(WINDOW), "--batch-size", str(B),
    "--kmax", "6","--elbow", "0.03"],                       # let it choose; MAX_BANDS=8 supports up to 6
    cwd=CODE, capture_output=True, text=True)
print(r.stdout)
if r.returncode: print("STDERR:\n", r.stderr[-1500:]); raise RuntimeError("derive failed")

bc = json.load(open(os.path.join(SH, "band_config.json")))
cutoffs = bc["internal_cutoffs"]          # a LIST — any length now
print(f"\nDERIVED: k={bc['k_final']} length bands (+chunked) -> {len(cutoffs)+2} total")
print(f"cutoffs: {cutoffs}")
print(f"populations: {bc['band_populations']} + chunked {bc['chunked_population']}")
print(f"weights: {[round(w,3) for w in bc['category_weights']]} (sum={sum(bc['category_weights']):.3f})")

elbow k=5  window-capped k=5  final k=5 (+chunked)
band edges (max_len per band): [163, 336, 558, 775, 1024]
internal cutoffs -> Config:    [163, 336, 558, 775]
band populations:              [98346, 73039, 66879, 46802, 44803]
category weights:              [0.214, 0.159, 0.145, 0.102, 0.097, 0.283]
padding curve (k: tokens):     {1: 209805220, 2: 90829210, 3: 55178825, 4: 41085980, 5: 32059170, 6: 26499404}

wrote /content/sft/sft_final/_shards/band_config.json


DERIVED: k=5 length bands (+chunked) -> 6 total
cutoffs: [163, 336, 558, 775]
populations: [98346, 73039, 66879, 46802, 44803] + chunked 129967
weights: [0.214, 0.159, 0.145, 0.102, 0.097, 0.283] (sum=1.000)


## 4 - make() : build the pipeline\nCutoffs applied at LOAD time via pc.band_short_max/band_medium_max. No shard rebuild to change them.

In [11]:
import os, glob, json, sys
sys.path.insert(0, CODE)
import uds_loader

def find_shards_dir():
    for root, dirs, files in os.walk("/content"):
        dirs[:] = [d for d in dirs if d != ".git"]
        if "meta.json" in files and glob.glob(os.path.join(root, "shard_*.bin")):
            return root
    return None

SH = find_shards_dir(); assert SH
meta   = json.load(open(os.path.join(SH, "meta.json")))
MAXLEN = int(meta["max_seq_length"]); B, WINDOW = 32, 4
_bc = os.path.join(SH, "band_config.json")
BC = json.load(open(_bc)) if os.path.exists(_bc) else None
if BC: print("derived band config: k=", BC["k_final"], "cutoffs=", BC["internal_cutoffs"])

def make(baseline, band_cutoffs=None):
    # cutoffs: list of k-1 ascending ints. band count = len(cutoffs)+2 (+chunked)
    if band_cutoffs is None:
        band_cutoffs = BC["internal_cutoffs"] if BC else [512, 1536]
    band_cutoffs = [int(c) for c in band_cutoffs]
    n_bands = len(band_cutoffs) + 2          # k length bands + chunked

    pc = uds_loader.PipelineConfig()
    pc.shards = sorted(glob.glob(os.path.join(SH, "shard_*.bin")))
    pc.num_epochs = 1; pc.resident_window = WINDOW; pc.B = B
    pc.baseline = baseline; pc.pad_id = int(meta["pad_id"])
    pc.option_b_window = MAXLEN; pc.pad_to_multiple = 8
    pc.prefetch_workers = 3; pc.ring_capacity = 4

    # load-time bands (no shard rebuild needed to change these)
    pc.band_cutoffs     = band_cutoffs
    pc.band_max_seq_len = int(MAXLEN)

    if not baseline:
        # one single-band category per band; chunked is the LAST one
        pc.profile_bands      = [[b, b] for b in range(n_bands)]
        pc.profile_mix        = [[1.0, 0.0]] * n_bands
        pc.profile_is_chunked = [False] * (n_bands - 1) + [True]
    return uds_loader.DataPipeline(pc)

print("make() ready — default cutoffs:",
      BC["internal_cutoffs"] if BC else [512, 1536])


derived band config: k= 5 cutoffs= [163, 336, 558, 775]
make() ready — default cutoffs: [163, 336, 558, 775]


## 5 - Padding comparison: default vs derived cutoffs\nNo shard rebuild between the two runs.

In [12]:
import sys, importlib
sys.path.insert(0, CODE)
import loader_only_loop; importlib.reload(loader_only_loop)
from loader_only_loop import run_loader_only

p = make(False, band_cutoffs=bc["internal_cutoffs"]); p.start()
try:    after, _ = run_loader_only(p, sim_step_s=0.0, log_every=10000)
finally: p.stop()
print(f"AFTER  (derived) : padding={after['padding_pct']:.2f}%  batches={after['batches']}")


[   2.85s] batch= 10000 samples=  291678 wait= 2.743s sim=  0.00s | batches/s=  3503.3 streamed=294912 done=False
AFTER  (derived) : padding=10.60%  batches=15667


## 6 - Proxy training-time estimate (optional, GPU)

In [13]:
import torch, gc, importlib
import proxy_epoch_time; importlib.reload(proxy_epoch_time)
from proxy_epoch_time import calibrate_A_C, estimate_epoch_seconds
from realtime_train_loop import TinyDecoder

gc.collect(); torch.cuda.empty_cache()
device = "cuda" if torch.cuda.is_available() else "cpu"
VOCAB = int(meta.get("vocab_size", 32000))
model = TinyDecoder(vocab_size=VOCAB, d_model=512, n_layers=6, n_heads=8, d_ff=2048, max_len=MAXLEN)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4)

cal_B = 8; widths = [w for w in (128,256,512,768,1024) if w <= MAXLEN]

# NOTE: now returns FOUR values — D comes first
D, A, C, table = calibrate_A_C(model, opt, B=cal_B, widths=widths, vocab_size=VOCAB,
                               device=device, ignore_index=int(meta.get("ignore_index",-100)),
                               use_amp=True, warmup=3, reps=4, pad_id=int(meta["pad_id"]))

p = make(False); p.start()
try:    summary, per_batch = estimate_epoch_seconds(p, A, C, D, log_every=2000)   # pass D
finally: p.stop()

import json; print(json.dumps(summary, indent=2))
print(f"\nESTIMATED FULL-EPOCH TRAIN TIME: {summary['estimated_train_hours']:.2f} h")

/content/sft/sft_final/realtime_train_loop.py:43: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.blocks = nn.TransformerEncoder(layer, num_layers=n_layers)
/content/sft/sft_final/proxy_epoch_time.py:35: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp and is_cuda)


  width T=  128  ->     75.81 ms/step
  width T=  256  ->    107.06 ms/step
  width T=  512  ->    182.01 ms/step
  width T=  768  ->    271.29 ms/step
  width T= 1024  ->    363.77 ms/step

  fit: time = 0.0436 + 2.907e-05*(B*T) + 9.892e-09*(B*T^2)
       fixed overhead D = 43.6 ms/step | max residual = 2.41 ms
[  0.56s] batch=  2000 est_epoch_so_far= 0.432 h streamed=62738 done=False
[  0.88s] batch=  4000 est_epoch_so_far= 0.823 h streamed=119528 done=False
[  1.19s] batch=  6000 est_epoch_so_far= 1.225 h streamed=175560 done=False
[  1.47s] batch=  8000 est_epoch_so_far= 1.605 h streamed=227691 done=False
[  1.74s] batch= 10000 est_epoch_so_far= 1.989 h streamed=285208 done=False
[  2.01s] batch= 12000 est_epoch_so_far= 2.363 h streamed=340378 done=False
[  2.29s] batch= 14000 est_epoch_so_far= 2.739 h streamed=396618 done=False
[  2.57s] batch= 16000 est_epoch_so_far= 3.137 h streamed=453667 done=False
{
  "batches": 16337,
  "samples": 459836,
  "padded_tokens": 291865376,
  "rea